# F5-probability — Session 01: Random Variables and Expectation

**Session length:** about 80 minutes • **Concepts:** random-variables,
expectation (with sampling-simulation used throughout) — the language for
reasoning about processes whose outcomes vary.

Checkpoint answers are collected at the end of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Randomness you can rerun

Probability is the mathematics of processes whose outcome you cannot predict
run by run — a die roll, a shuffled deck, a noisy measurement — but whose
*long-run behavior* is completely predictable. This unit works in both
registers at once:

- **Simulation:** run the process many times in NumPy and *measure* what
  happens.
- **Algebra:** compute what *must* happen, exactly, with pencil and paper.

Every claim we prove will also be checked by simulation, and every simulation
follows the F1 seeding discipline: one `SEED` constant, one generator, every
draw from it — so your numbers and this notebook's numbers agree exactly.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

rolls = rng.integers(1, 7, size=10)     # ten fair-die rolls, faces 1..6
print(rolls)

Run it twice from a fresh generator and you get the same ten faces — that is
what makes a simulation *checkable*. Throughout this unit, "simulate the
process" always means: seeded generator, one array of many repetitions, no
loops.

### Checkpoint 1

1. Simulate 20 flips of a fair coin (0 = tails, 1 = heads) with a generator
   seeded from `SEED`. Which draw method from F1 fits, and why?
2. Your friend simulates the same 20 flips but forgot the seed. Name the two
   things that go wrong when someone re-runs their notebook.

## 2. Random variables

A **random variable** is a *number produced by a random process*. Not the
process itself, not a description — a number, so we can do arithmetic on it.

- Roll a die; let $X$ be the face shown. $X$ is a random variable with
  possible values $1, 2, 3, 4, 5, 6$.
- Flip three coins; let $H$ be the number of heads. Possible values
  $0, 1, 2, 3$.
- Roll two dice; let $T$ be the total. Possible values $2$ through $12$.

The convention: capital letters ($X$, $Y$, $T$) name random variables; small
letters ($x$, $y$) name the particular values they might take. "$X = 5$" is an
*event* — it happens on some runs and not others.

One run of the process produces one value of $X$. A simulation array of
10,000 entries is 10,000 independent runs — one value of $X$ per entry. Here
is a random variable that is *not* uniform: a spinner whose pointer lands on
1 half the time, on 2 a quarter of the time, and on 5 a quarter of the time.
`rng.choice` with the `p=` argument simulates it directly:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

values = np.array([1, 2, 5])
probs = np.array([0.5, 0.25, 0.25])
spins = rng.choice(values, size=12, p=probs)
print(spins)

### Checkpoint 2

1. For the three-coin experiment, list every possible value of $H$ (number of
   heads) and simulate 10 runs of $H$ — one array expression for the flips
   (hint: a `(10, 3)` array of 0/1 draws) and one axis aggregation.
2. Which of these are random variables, and which are not? (i) the total of
   two dice; (ii) the event "the total is 7"; (iii) the number of sixes in
   100 rolls; (iv) the number 3.5.

## 3. Distribution tables

A discrete random variable is completely described by its **distribution
table**: every possible value $x$ together with its probability
$P(X = x)$. For the spinner above:

| $x$ | 1 | 2 | 5 |
| --- | --- | --- | --- |
| $P(X=x)$ | $\tfrac12$ | $\tfrac14$ | $\tfrac14$ |

Two rules make a table legitimate:

1. every probability is between 0 and 1;
2. the probabilities **sum to exactly 1** — some outcome must happen.

Probabilities of *composite* events come from adding table entries:
$P(X \ge 2) = P(X{=}2) + P(X{=}5) = \tfrac12$, and the **complement rule**
$P(X \ne 1) = 1 - P(X{=}1)$.

**Simulation meets the table.** The *empirical frequency* of a value is the
fraction of runs that produced it — a mask's `.mean()`, exactly as in F1. As
the number of runs grows, frequencies settle toward the table's
probabilities:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

values = np.array([1, 2, 5])
probs = np.array([0.5, 0.25, 0.25])

for_n = {}
spins = rng.choice(values, size=100_000, p=probs)
for n in (100, 1_000, 100_000):
    part = spins[:n]
    for_n[n] = [(part == v).mean() for v in values]

print("value:              1      2      5")
for n, freqs in for_n.items():
    print(f"n = {n:>7,}:  ", "  ".join(f"{f:.3f}" for f in freqs))
print("exact:            0.500  0.250  0.250")

(The list comprehension above is display plumbing, not the simulation — every
probability estimate is a loop-free mask mean.)

At $n = 100$ the frequencies wobble visibly; at $n = 100{,}000$ they sit
within a few thousandths of the exact probabilities. This pattern — *observe
frequencies, trust them more as $n$ grows* — is the engine of the whole unit:
we will use big-$n$ simulation to verify every algebraic claim we make.

### Checkpoint 3

1. A claimed distribution table has probabilities $0.5,\ 0.3,\ 0.3$ over
   three values. What is wrong, and which rule does it break?
2. Using the seeded `spins` array above, estimate $P(X \ge 2)$ two ways: with
   one mask directly, and via the complement rule from the frequency of 1.
   Confirm the two estimates agree exactly.

## 4. Expectation

Play the spinner for money: you win the number shown, in tokens. What is a
fair price for one spin? Over $4n$ spins you expect about $2n$ ones, $n$
twos, and $n$ fives — total winnings about $2n + 2n + 5n = 9n$ tokens, so
$\tfrac{9}{4} = 2.25$ per spin. That per-run long-run average is the
**expectation** (or *expected value*) of $X$:

$$E[X] \;=\; \sum_x x \cdot P(X = x)$$

— each possible value times its probability, summed over the table. For the
spinner: $E[X] = 1 \cdot \tfrac12 + 2 \cdot \tfrac14 + 5 \cdot \tfrac14
= 2.25$. For a fair die: $E[X] = \tfrac{1+2+3+4+5+6}{6} = 3.5$.

Two readings of the same number:

- **Algebraic:** a probability-weighted sum over the *table* — no randomness
  left in it; $E[X]$ is a constant.
- **Long-run:** the value the *sample mean* of many runs settles toward.

In code, the algebraic side is an indexed sum $\sum_i x_i p_i$ — elementwise
product, then `.sum()`:

In [ ]:
values = np.array([1.0, 2.0, 5.0])
probs = np.array([0.5, 0.25, 0.25])
exact = (values * probs).sum()          # sum_i  x_i * p_i
print("E[X] exact:", exact)

SEED = 20260804
rng = np.random.default_rng(SEED)
spins = rng.choice(values, size=100_000, p=probs)
for n in (100, 1_000, 100_000):
    print(f"sample mean of first {n:>7,}: {spins[:n].mean():.4f}")

The sample mean drifts toward 2.25 as $n$ grows — simulation confirming
algebra. Note what $E[X]$ is *not*: it is not a value the spinner can show
(no face says 2.25), and it is not what happens on any single run. It is the
long-run per-run average.

### Checkpoint 4

1. By hand: a raffle ticket pays 100 tokens with probability $0.01$ and 0
   otherwise. Compute $E[X]$. Would you pay 2 tokens for a ticket?
2. Compute the fair die's $E[X] = 3.5$ by simulation: seeded generator,
   100,000 rolls, sample mean. How far off is the estimate?

## 5. Functions of a random variable

If $X$ is a random variable, so is any function of it: $X^2$, $3X + 2$,
$|X - 3|$. The process runs, $X$ comes out, the function is applied — a
number, run by run.

To compute the expectation of $g(X)$ you do **not** need a new table — apply
$g$ to each value, keep the probabilities:

$$E[g(X)] \;=\; \sum_x g(x)\, P(X = x)$$

For the spinner, $E[X^2] = 1^2 \cdot \tfrac12 + 2^2 \cdot \tfrac14 + 5^2
\cdot \tfrac14 = 0.5 + 1 + 6.25 = 7.75$.

**The order of operations trap.** $E[X^2]$ and $(E[X])^2$ are different
numbers: $7.75$ versus $2.25^2 = 5.0625$. *Apply the function first, then
average* is not the same as *average first, then apply the function* —
squaring exaggerates the large value 5 before the averaging can dilute it.
The gap between these two numbers is no accident; it becomes the star of
Session 02. Simulation agrees:

In [ ]:
values = np.array([1.0, 2.0, 5.0])
probs = np.array([0.5, 0.25, 0.25])
print("E[X^2] exact:   ", ((values ** 2) * probs).sum())
print("(E[X])^2 exact: ", ((values * probs).sum()) ** 2)

SEED = 20260804
rng = np.random.default_rng(SEED)
spins = rng.choice(values, size=100_000, p=probs)
print("mean of squares:", (spins ** 2).mean())      # estimates E[X^2]
print("square of mean: ", spins.mean() ** 2)        # estimates (E[X])^2

### Checkpoint 5

1. For a fair die, compute $E[X^2]$ by hand as an exact fraction, and check
   it against a seeded simulation.
2. Let $g(X) = |X - 3|$ for the fair die. Compute $E[g(X)]$ by hand from the
   table, and explain why it differs from $|E[X] - 3| = 0.5$.

## 6. Linearity of expectation

Expectation plays perfectly with sums and scaling. For any random variables
$X, Y$ and constants $a, b$:

$$E[aX + b] = a\,E[X] + b \qquad\text{and}\qquad E[X + Y] = E[X] + E[Y].$$

**Proof of the first** (discrete, from the definition):

$$E[aX + b] = \sum_x (ax + b)\,P(X{=}x)
 = a \underbrace{\sum_x x\,P(X{=}x)}_{E[X]}
 + b \underbrace{\sum_x P(X{=}x)}_{1} = a\,E[X] + b. \; \blacksquare$$

**Proof of the second** (discrete): let $P(x, y)$ be the joint probability
that $X{=}x$ and $Y{=}y$ at once. Summing over all pairs,

$$E[X + Y] = \sum_{x}\sum_{y} (x + y)\,P(x, y)
 = \sum_{x}\sum_{y} x\,P(x, y) + \sum_{x}\sum_{y} y\,P(x, y)
 = E[X] + E[Y], $$

because $\sum_y P(x, y) = P(X{=}x)$ (for fixed $x$, the pairs $(x, y)$ split
the event $X{=}x$ by cases) and likewise with roles swapped. $\blacksquare$

**The remarkable part:** the proof never assumed anything about how $X$ and
$Y$ are related. Linearity holds even when they influence each other. Watch
it hold for a *blatantly* dependent pair — a die $X$ and $Y = 7 - X$ (the
opposite face):

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

x = rng.integers(1, 7, size=100_000).astype(np.float64)
y = 7.0 - x                                  # completely determined by x
print("E[X] est:    ", x.mean())
print("E[Y] est:    ", y.mean())
print("E[X+Y] est:  ", (x + y).mean(), "   (exactly 7 every run)")
print("E[X]+E[Y]:   ", x.mean() + y.mean())

Two-dice total, expected value: $E[T] = E[X_1] + E[X_2] = 3.5 + 3.5 = 7$ —
no 36-entry table needed. That shortcut scales: the expected total of 100
dice is $350$, one line of linearity instead of an astronomically large
table.

### Checkpoint 6

1. Tickets cost 2 tokens and pay the spinner value ($E[X] = 2.25$). Using
   linearity, what is the expected *net gain* $E[X - 2]$ per play? For 1,000
   plays?
2. Prove from the definition that $E[c] = c$ for a constant $c$ (a "random"
   variable with one possible value), then use linearity to explain why
   $E[X - E[X]] = 0$ for every random variable $X$.

## 7. Worked exam-style example

Multiple-choice items in this unit's register give five options and expect
hand computation. Work it before reading the solution.

> A game pays $G$ tokens: with probability $0.2$ you win 10, with
> probability $0.3$ you win 2, otherwise you win nothing. An entry ticket
> costs 3 tokens. What is the expected net gain $E[G - 3]$ per play?
>
> (A) $-1.4$  (B) $-0.4$  (C) $0$  (D) $2.6$  (E) $1.2$

Step by step:

1. Table of $G$: values $10, 2, 0$ with probabilities $0.2, 0.3, 0.5$ (the
   "otherwise" row must bring the total to 1).
2. $E[G] = 10(0.2) + 2(0.3) + 0(0.5) = 2 + 0.6 = 2.6$.
3. Linearity: $E[G - 3] = E[G] - 3 = -0.4$. **Answer (B).**

The traps are real options: forgetting the ticket gives (D); treating the
three outcomes as equally likely gives $E[G] = 4$, net $1$ — near (E);
mis-reading "otherwise" as probability $0.3$ breaks rule 2 of Section 3.
Verify by simulation (allowed here, not in the exam room):

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
g = rng.choice(np.array([10.0, 2.0, 0.0]), size=200_000, p=np.array([0.2, 0.3, 0.5]))
print("E[G - 3] est:", (g - 3.0).mean())

### Checkpoint 7

1. Exam-style, by hand: $X$ takes values $-2, 1, 4$ with probabilities
   $\tfrac14, \tfrac12, \tfrac14$. $E[3X + 1] =$ (A) $1$ (B) $2.5$
   (C) $4$ (D) $4.75$ (E) $5.5$. Show your steps.
2. Build the simulation check for your answer: seeded generator, 100,000
   draws, one mean.

## 8. Common pitfalls I

**Pitfall — reading the sample mean as the expectation.** The sample mean
*estimates* $E[X]$; it is not equal to it. At small $n$ the estimate can be
badly off, and every reseed gives a different wrong number:

In [ ]:
SEED = 20260804
values = np.array([1.0, 2.0, 5.0])
probs = np.array([0.5, 0.25, 0.25])

means_n10 = np.array([np.random.default_rng(SEED + k).choice(values, size=10, p=probs).mean()
                      for k in range(6)])
print("six 10-spin sample means:", means_n10)      # spread all over
print("E[X] is exactly 2.25 regardless")

The fix: say "the sample mean was 2.31" for measurements and reserve
"$E[X]$" for the table computation, and never trust a 10-run estimate — the
checkpoint answers use $n$ of 100,000 for a reason.

**Pitfall — averaging before applying the function.** $E[X^2] \ne (E[X])^2$
(Section 5: $7.75$ vs $5.0625$). The broken pattern is computing a summary
first and pushing it through the function afterward; apply the function to
each value, *then* take the probability-weighted sum.

**Pitfall — expecting $E[X]$ to be a possible outcome.** A fair die "expects"
3.5, a face it cannot show. Expectation is a long-run average, not a
prediction of any single run; "the most likely value" is a different concept
(the *mode*).

### Checkpoint 8

1. A classmate simulates 10 spins, gets a sample mean of 2.9, and reports
   "$E[X] = 2.9$." Name both mistakes in that sentence.
2. True or false, with one line of justification each: (i) $E[X]$ is always
   one of $X$'s possible values; (ii) $E[X^2] = (E[X])^2$ for every $X$;
   (iii) $E[X + Y] = E[X] + E[Y]$ requires $X$ and $Y$ to be unrelated.

## Checkpoint answers

### Checkpoint 1 answers

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
flips = rng.integers(0, 2, size=20)       # 1. integers(0, 2) draws 0 or 1
print(flips)

# 2. Nobody can reproduce their exact flips (different numbers every run), so
#    their reported counts cannot be verified — and any bug they hit cannot be
#    reproduced to debug it.

### Checkpoint 2 answers

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
flips = rng.integers(0, 2, size=(10, 3))   # 1. ten runs, three coins each
h = flips.sum(axis=1)                      #    possible values of H: 0, 1, 2, 3
print(h)

# 2. Random variables: (i) and (iii) — numbers produced by a random process.
#    (ii) is an event (it happens or not; the related NUMBER would be the 0/1
#    indicator of the event). (iv) is a constant, not produced by any process
#    (at best a "random" variable with one possible value).

### Checkpoint 3 answers

In [ ]:
# 1. 0.5 + 0.3 + 0.3 = 1.1 > 1 — breaks rule 2 (probabilities must sum to
#    exactly 1), so it is not a distribution table.

SEED = 20260804                            # 2.
rng = np.random.default_rng(SEED)
values = np.array([1, 2, 5])
spins = rng.choice(values, size=100_000, p=np.array([0.5, 0.25, 0.25]))
direct = (spins >= 2).mean()
via_complement = 1.0 - (spins == 1).mean()
print(direct, via_complement, direct == via_complement)

### Checkpoint 4 answers

In [ ]:
# 1. E[X] = 100(0.01) + 0(0.99) = 1 token. A 2-token ticket costs twice its
#    long-run value — expected net gain 1 - 2 = -1 per ticket, so no.

SEED = 20260804                            # 2.
rng = np.random.default_rng(SEED)
rolls = rng.integers(1, 7, size=100_000)
print("estimate:", rolls.mean(), "  gap:", abs(rolls.mean() - 3.5))

### Checkpoint 5 answers

In [ ]:
# 1. E[X^2] = (1 + 4 + 9 + 16 + 25 + 36) / 6 = 91/6 ≈ 15.1667.
SEED = 20260804
rng = np.random.default_rng(SEED)
rolls = rng.integers(1, 7, size=100_000)
print("simulated E[X^2]:", (rolls ** 2).mean(), "  exact:", 91 / 6)

# 2. E[|X - 3|] = (2 + 1 + 0 + 1 + 2 + 3) / 6 = 9/6 = 1.5.
#    |E[X] - 3| = 0.5 applies the function AFTER averaging; the deviations
#    partly cancel inside the mean, while E[|X-3|] averages sizes that cannot
#    cancel. Function-then-average and average-then-function differ.

### Checkpoint 6 answers

In [ ]:
# 1. E[X - 2] = E[X] - 2 = 0.25 tokens per play; over 1,000 plays the
#    expected total is 1,000 * 0.25 = 250 tokens (linearity again: the
#    expectation of a sum of 1,000 per-play gains is the sum of expectations).

# 2. A constant c has the one-row table P(X = c) = 1, so
#    E[c] = c * 1 = c. Then, treating the number E[X] as that kind of
#    constant, linearity gives
#    E[X - E[X]] = E[X] - E[E[X]] = E[X] - E[X] = 0.
print("see comments")

### Checkpoint 7 answers

In [ ]:
# 1. E[X] = (-2)(1/4) + 1(1/2) + 4(1/4) = -0.5 + 0.5 + 1 = 1.
#    E[3X + 1] = 3 * 1 + 1 = 4  ->  answer (C).

SEED = 20260804                            # 2.
rng = np.random.default_rng(SEED)
x = rng.choice(np.array([-2.0, 1.0, 4.0]), size=100_000, p=np.array([0.25, 0.5, 0.25]))
print("E[3X + 1] est:", (3 * x + 1).mean())

### Checkpoint 8 answers

In [ ]:
# 1. (a) 2.9 is a sample-mean ESTIMATE, not E[X] — the expectation is the
#    exact table computation 2.25. (b) n = 10 is far too small for the
#    estimate to be trusted anyway (Section 8 showed 10-spin means scattered
#    widely around 2.25).

# 2. (i) False — the fair die's E[X] = 3.5 is not a face.
#    (ii) False — spinner: E[X^2] = 7.75 but (E[X])^2 = 5.0625.
#    (iii) False — linearity's proof never used any relation between X and Y;
#          it holds for dependent pairs too (the X and 7 - X demo).
print("see comments")